# CrewAI: One Agent, Three Tools

CrewAI agents become genuinely useful once they can *do* things beyond generating text — check
live data, run a calculation, query a database. That capability comes from **tools**: Python
functions or classes an agent can call, get a result back from, and reason over.

This notebook builds a single agent and gives it three tools:

1. **Weather tool** — look up the current weather for a city via a live weather API.
2. **Currency converter tool** — convert an amount between two currencies.
3. **Employee database tool** — look up employee records (age, department, salary, years of
   employment) from a local SQLite database, with flexible filters.

**What this teaches:**

- The core CrewAI concepts — `Agent`, `Task`, `Crew` — applied to a single agent.
- How CrewAI tools work (`@tool` decorator vs. subclassing `BaseTool`).
- How a single agent decides *which* tool to call, and *when* to call more than one, for a
  single request — this is the ReAct-style "think → act → observe" loop under the hood.
- How to wrap something stateful (a SQLite DB) as a safe, structured tool input instead of
  letting the LLM write raw SQL.

This notebook is self-contained — just run the cells top to bottom.

## 1. Setup

Create a `.env` file in the same folder as this notebook with your key:

```
OPENAI_API_KEY=sk-...
```

`load_dotenv()` reads that file into the environment, so nothing sensitive gets typed into the
notebook itself.

In [1]:
%pip install crewai crewai-tools requests python-dotenv

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
            "langchain-oracledb", "langchain-text-splitters", 
            "llama-index", "langgraph","crewai", "crewai-tools"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-oracledb version: 1.5.0
langchain-text-splitters version: 1.1.2
llama-index version: 0.14.21
langgraph version: 1.2.4
crewai version: 1.15.17
crewai-tools version: 1.15.17


In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()  # reads .env in the current directory into the environment

# Fallback for anyone who hasn't created a .env file yet.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY not found in .env - enter it here: ")

print("Key set:", bool(os.environ.get("OPENAI_API_KEY")))

Key set: True


## 2. Two quick ways to build a tool

CrewAI gives you two patterns:

- **`@tool` decorator** — fastest way to turn a plain Python function into a tool. Best for
  simple, single-input tools. We'll use this for the weather and currency tools.
- **Subclass `BaseTool`** — more control: a Pydantic `args_schema` for multiple/typed/optional
  inputs, and a proper `_run()` method. Best when a tool takes several parameters or needs setup
  (like a DB connection). We'll use this for the employee-lookup tool.

In both cases, the **docstring / `description`** is what the LLM reads to decide *when* to use
the tool — write it like you're briefing a new hire, not documenting code for other engineers.

## 3. Tool 1 — Weather (live API)

We call **Open-Meteo** — a free weather API that needs no signup or API key, which keeps the
classroom setup friction-free. It's a real two-step API call:

1. **Geocode** the city name to latitude/longitude (Open-Meteo's geocoding endpoint).
2. **Fetch current conditions** for those coordinates (Open-Meteo's forecast endpoint, with
   `current_weather=True`).

The forecast API returns a numeric WMO weather code instead of a text description, so we map a
few common codes to readable text ourselves.

*(If you'd rather use a different provider — e.g. OpenWeatherMap — only the request/response
handling inside `get_weather` changes; the `@tool` wrapper and the rest of the notebook don't.)*

In [3]:
import requests

from crewai.tools import tool

GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# A subset of WMO weather interpretation codes - enough for a classroom demo.
# Full table: https://open-meteo.com/en/docs
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow fall", 73: "Moderate snow fall", 75: "Heavy snow fall",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


@tool("Weather Lookup Tool")
def get_weather(city: str) -> str:
    """Get the current weather conditions and temperature (Celsius) for a given city name,
    using a live weather API. Input should be a city name, e.g. 'Tokyo' or 'Mumbai'."""
    geo_resp = requests.get(GEOCODE_URL, params={"name": city, "count": 1}, timeout=10)
    geo_resp.raise_for_status()
    matches = geo_resp.json().get("results")
    if not matches:
        return f"Could not find a location matching '{city}'."

    place = matches[0]
    lat, lon = place["latitude"], place["longitude"]
    label = f"{place['name']}, {place.get('country', '')}".rstrip(", ")

    forecast_resp = requests.get(
        FORECAST_URL,
        params={"latitude": lat, "longitude": lon, "current_weather": True},
        timeout=10,
    )
    forecast_resp.raise_for_status()
    current = forecast_resp.json()["current_weather"]

    condition = WMO_CODES.get(current["weathercode"], "Unknown conditions")
    return (
        f"{label}: {condition}, {current['temperature']}°C, "
        f"wind {current['windspeed']} km/h."
    )


# Quick manual test, outside the agent, before we trust it to an LLM.
print(get_weather.run(city="Tokyo"))

Tokyo, Japan: Overcast, 24.6°C, wind 2.3 km/h.


In [4]:
print(get_weather.run(city="Delhi"))

Delhi, India: Partly cloudy, 32.8°C, wind 3.9 km/h.


## 4. Tool 2 — Currency converter

Same pattern: a mock fixed-rate table, deterministic and offline for the tutorial. In production
you'd call a live FX API (e.g. exchangerate.host, Open Exchange Rates) inside this function.

In [5]:
@tool("Currency Converter Tool")
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another. Provide amount as a number and
    currencies as 3-letter ISO codes, e.g. from_currency='USD', to_currency='INR'."""
    # Mock exchange rates, all relative to 1 USD.
    rates_to_usd = {"USD": 1.0, "INR": 83.0, "EUR": 0.92, "GBP": 0.79, "JPY": 149.0}

    from_c, to_c = from_currency.upper(), to_currency.upper()
    if from_c not in rates_to_usd or to_c not in rates_to_usd:
        supported = ", ".join(rates_to_usd)
        return f"Unsupported currency. Supported codes: {supported}."

    usd_amount = amount / rates_to_usd[from_c]
    converted = usd_amount * rates_to_usd[to_c]
    return f"{amount} {from_c} = {converted:.2f} {to_c}"


print(convert_currency.run(amount=100, from_currency="USD", to_currency="INR"))

100.0 USD = 8300.00 INR


## 5. Tool 3 — Employee database lookup (SQLite)

First, create and seed a small local SQLite database — this stands in for "the company HR
database" the agent needs to query.

In [6]:
import sqlite3

DB_PATH = "employees.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS employees")
cur.execute("""
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        department TEXT NOT NULL,
        age INTEGER NOT NULL,
        salary INTEGER NOT NULL,
        years_of_employment REAL NOT NULL
    )
""")

employees = [
    (1, "Asha Rao",        "Engineering", 29, 95000, 3.5),
    (2, "Liam Chen",       "Engineering", 34, 112000, 6.0),
    (3, "Priya Nair",      "Sales",       41, 88000, 9.0),
    (4, "Marcus Webb",     "Sales",       26, 61000, 1.5),
    (5, "Sofia Garcia",    "HR",          38, 72000, 7.0),
    (6, "Daniel Kim",      "Engineering", 45, 138000, 14.0),
    (7, "Fatima Sheikh",   "Marketing",   31, 79000, 4.0),
    (8, "Tom Becker",      "Marketing",   50, 91000, 20.0),
    (9, "Emeka Okafor",    "HR",          27, 58000, 2.0),
    (10, "Nina Petrova",   "Engineering", 37, 121000, 8.5),
]

cur.executemany(
    "INSERT INTO employees (id, name, department, age, salary, years_of_employment) VALUES (?, ?, ?, ?, ?, ?)",
    employees,
)
conn.commit()
conn.close()

print(f"Seeded {len(employees)} employee records into {DB_PATH}")

Seeded 10 employee records into employees.db


Now the tool itself. We deliberately do **not** let the LLM write raw SQL — instead we
expose a small set of typed, optional filter fields via a Pydantic `args_schema`, and build a
safe parameterized query from them. This is the pattern you want for any tool that touches a
real database: constrain what the model can express so it can't do anything you didn't intend.

In [7]:
from typing import Optional, Type

from crewai.tools import BaseTool
from pydantic import BaseModel, Field


class EmployeeLookupInput(BaseModel):
    """Filters for searching the employees table. Leave any field unset to not filter on it."""

    name_contains: Optional[str] = Field(None, description="Substring to match against employee name (case-insensitive).")
    department: Optional[str] = Field(None, description="Exact department name, e.g. 'Engineering', 'Sales', 'HR', 'Marketing'.")
    min_salary: Optional[int] = Field(None, description="Minimum salary, inclusive.")
    max_salary: Optional[int] = Field(None, description="Maximum salary, inclusive.")
    min_years: Optional[float] = Field(None, description="Minimum years of employment, inclusive.")


class EmployeeLookupTool(BaseTool):
    name: str = "Employee Database Lookup Tool"
    description: str = (
        "Search the company employees SQLite database. Supports optional filters: "
        "name_contains, department, min_salary, max_salary, min_years. "
        "Returns matching employees with their age, department, salary, and years of employment. "
        "Call with no filters to list everyone."
    )
    args_schema: Type[BaseModel] = EmployeeLookupInput
    db_path: str = DB_PATH

    def _run(
        self,
        name_contains: Optional[str] = None,
        department: Optional[str] = None,
        min_salary: Optional[int] = None,
        max_salary: Optional[int] = None,
        min_years: Optional[float] = None,
    ) -> str:
        query = "SELECT name, department, age, salary, years_of_employment FROM employees WHERE 1=1"
        params: list = []

        if name_contains:
            query += " AND LOWER(name) LIKE ?"
            params.append(f"%{name_contains.lower()}%")
        if department:
            query += " AND department = ?"
            params.append(department)
        if min_salary is not None:
            query += " AND salary >= ?"
            params.append(min_salary)
        if max_salary is not None:
            query += " AND salary <= ?"
            params.append(max_salary)
        if min_years is not None:
            query += " AND years_of_employment >= ?"
            params.append(min_years)

        conn = sqlite3.connect(self.db_path)
        try:
            rows = conn.execute(query, params).fetchall()
        finally:
            conn.close()

        if not rows:
            return "No employees matched those filters."

        lines = [
            f"- {name}: {department}, age {age}, salary ${salary:,}, {years} yrs employed"
            for name, department, age, salary, years in rows
        ]
        return "\n".join(lines)


employee_tool = EmployeeLookupTool()

# Quick manual test: Retrieve all emmployees who dept=Engg and min salary is 100,000
print(employee_tool.run(department="Engineering", min_salary=100000))

- Liam Chen: Engineering, age 34, salary $112,000, 6.0 yrs employed
- Daniel Kim: Engineering, age 45, salary $138,000, 14.0 yrs employed
- Nina Petrova: Engineering, age 37, salary $121,000, 8.5 yrs employed


## 6. Build the single agent

One agent, three tools. The `role`/`goal`/`backstory` matter more here than in a multi-agent
crew — they're the only steering you give the model on *how* to combine tools for a request.

In [8]:
from crewai import Agent, Crew, Process, Task, LLM

llm = LLM(model="gpt-4o-mini", temperature=0.2)

assistant = Agent(
    role="Ops Assistant",
    goal="Answer questions accurately by using the weather, currency, and employee database tools whenever they're needed, instead of guessing",
    backstory=(
        "You are a meticulous operations assistant. You never make up numbers — "
        "if a question involves weather, currency conversion, or employee records, "
        "you always call the matching tool rather than relying on memory."
    ),
    tools=[get_weather, convert_currency, employee_tool],
    llm=llm,
    verbose=True,
)

## 7. Give it a task that needs all three tools

A single task with a multi-part question forces the agent to plan: figure out which tools are
relevant, call each one, then combine the results into one answer.

In [10]:
task = Task(
    description=(
        "Answer all three of the following, using your tools rather than guessing:\n"
        "1. What is the current weather in Tokyo?\n"
        "2. Convert 500 USD to INR.\n"
        "3. List Engineering employees earning at least 100000, with their years of employment.\n"
        "Present the answer as three short, clearly labeled sections."
    ),
    expected_output="Three labeled sections (Weather, Currency, Employees) answering each sub-question using tool results.",
    agent=assistant,
)

crew = Crew(agents=[assistant], tasks=[task], process=Process.sequential, verbose=True)

# result = crew.kickoff()
# crew.kickoff() is sync and errors inside Jupyter's already-running event loop.
# kickoff_async() + await is the fix — Jupyter cells support top-level await.

result = await crew.kickoff_async()  # This is for jupyter notebook

print("\n\n=== FINAL ANSWER ===\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 80b59ff0-65e9-4d1a-9d70-e1f04b91394a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer all three of the following, using your tools rather than guessing:                                │
│  1. What is the current weather in Tokyo?                                                                       │
│  2. Convert 500 USD to INR.                                                                                     │
│  3. List Engineering employees earning at least 100000, with their years of employment.                         │
│  Present the answer as three short, clearly labeled sections.                                                   │
│  ID: 2ff9d95d-71d7-4537-ba4b-fdbe1d640f07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ops Assistant                                                                                           │
│                                                                                                                 │
│  Task: Answer all three of the following, using your tools rather than guessing:                                │
│  1. What is the current weather in Tokyo?                                                                       │
│  2. Convert 500 USD to INR.                                                                                     │
│  3. List Engineering employees earning at least 100000, with their years of employment.                         │
│  Present the answer as three short, clearly labeled sections.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: weather_lookup_tool                                                                                      │
│  Args: {'city': 'Tokyo'}                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: currency_converter_tool                                                                                  │
│  Args: {'amount': 500, 'from_currency': 'USD', 'to_currency': 'INR'}                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ops Assistant                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Weather                                                                                                    │
│  Tokyo, Japan: Overcast, 24.5°C, wind 2.3 km/h.                                                                 │
│                                                                                                                 │
│  ### Currency                                                                                                   │
│  500.0 USD = 41500.00 INR                                                                                       │
│                                                                                                                 │
│  ### Employees                                                                                                  │
│  - Liam Chen: Engineering, age 34, salary $112,000, 6.0 yrs employed                                            │
│  - Daniel Kim: Engineering, age 45, salary $138,000, 14.0 yrs employed                                          │
│  - Nina Petrova: Engineering, age 37, salary $121,000, 8.5 yrs employed                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer all three of the following, using your tools rather than guessing:                                │
│  1. What is the current weather in Tokyo?                                                                       │
│  2. Convert 500 USD to INR.                                                                                     │
│  3. List Engineering employees earning at least 100000, with their years of employment.                         │
│  Present the answer as three short, clearly labeled sections.                                                   │
│  Agent: Ops Assistant                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL ANSWER ===

### Weather
Tokyo, Japan: Overcast, 24.5°C, wind 2.3 km/h.

### Currency
500.0 USD = 41500.00 INR

### Employees
- Liam Chen: Engineering, age 34, salary $112,000, 6.0 yrs employed
- Daniel Kim: Engineering, age 45, salary $138,000, 14.0 yrs employed
- Nina Petrova: Engineering, age 37, salary $121,000, 8.5 yrs employed


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 80b59ff0-65e9-4d1a-9d70-e1f04b91394a                                                                       │
│  Final Output: ### Weather                                                                                      │
│  Tokyo, Japan: Overcast, 24.5°C, wind 2.3 km/h.                                                                 │
│                                                                                                                 │
│  ### Currency                                                                                                   │
│  500.0 USD = 41500.00 INR                                                                                       │
│                                                                                                                 │
│  ### Employees                                                                                                  │
│  - Liam Chen: Engineering, age 34, salary $112,000, 6.0 yrs employed                                            │
│  - Daniel Kim: Engineering, age 45, salary $138,000, 14.0 yrs employed                                          │
│  - Nina Petrova: Engineering, age 37, salary $121,000, 8.5 yrs employed                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────────────────────────────────────────────────────────────── Tracing Preference Saved ───────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                                                                                                       │
│  Info: Tracing has been disabled.                                                                                                                                                                     │
│                                                                                                                                                                                                       │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                                                                                                       

**What to watch in the verbose log above:** the agent doesn't call all three tools blindly —
it reads the task, decides tool #1 is needed for the weather question, calls it, reads the
observation, moves to the currency question, calls tool #2, then calls the employee tool with
`department="Engineering"` and `min_salary=100000` — filters it inferred from plain English,
because those are exactly the fields we exposed in `EmployeeLookupInput`. That mapping from
natural language to structured arguments is the `args_schema` doing its job.

In [11]:
task = Task(
    description=(
        "Answer following, using your tools rather than guessing:\n"
        "Who has been employed the shortest, and what's their salary?\n"
        "Present the answer as three short, clearly labeled sections."
    ),
    expected_output="Three labeled sections (Weather, Currency, Employees) answering each sub-question using tool results.",
    agent=assistant,
)

crew = Crew(agents=[assistant], tasks=[task], process=Process.sequential, verbose=True)

# result = crew.kickoff()
# crew.kickoff() is sync and errors inside Jupyter's already-running event loop.
# kickoff_async() + await is the fix — Jupyter cells support top-level await.

result = await crew.kickoff_async()  # This is for jupyter notebook

print("\n\n=== FINAL ANSWER ===\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ef6f6bf1-27fd-46f4-aacc-da1dac4fa390                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer following, using your tools rather than guessing:                                                 │
│  Who has been employed the shortest, and what's their salary?                                                   │
│  Present the answer as three short, clearly labeled sections.                                                   │
│  ID: b281dd15-5561-4443-98a3-0cbf8548a23d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ops Assistant                                                                                           │
│                                                                                                                 │
│  Task: Answer following, using your tools rather than guessing:                                                 │
│  Who has been employed the shortest, and what's their salary?                                                   │
│  Present the answer as three short, clearly labeled sections.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: employee_database_lookup_tool                                                                            │
│  Args: {'name_contains': None, 'department': None, 'min_salary': None, 'max_salary': None, 'min_years': None}   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool employee_database_lookup_tool executed with result: - Asha Rao: Engineering, age 29, salary $95,000, 3.5 yrs employed
- Liam Chen: Engineering, age 34, salary $112,000, 6.0 yrs employed
- Priya Nair: Sales, age 41, salary $88,000, 9.0 yrs employed
- Ma...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ops Assistant                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Weather:**                                                                                                   │
│  N/A                                                                                                            │
│                                                                                                                 │
│  **Currency:**                                                                                                  │
│  N/A                                                                                                            │
│                                                                                                                 │
│  **Employees:**                                                                                                 │
│  The employee who has been employed the shortest is Marcus Webb, with a salary of $61,000. He has been          │
│  employed for 1.5 years.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer following, using your tools rather than guessing:                                                 │
│  Who has been employed the shortest, and what's their salary?                                                   │
│  Present the answer as three short, clearly labeled sections.                                                   │
│  Agent: Ops Assistant                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL ANSWER ===

**Weather:**  
N/A

**Currency:**  
N/A

**Employees:**  
The employee who has been employed the shortest is Marcus Webb, with a salary of $61,000. He has been employed for 1.5 years.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ef6f6bf1-27fd-46f4-aacc-da1dac4fa390                                                                       │
│  Final Output: **Weather:**                                                                                     │
│  N/A                                                                                                            │
│                                                                                                                 │
│  **Currency:**                                                                                                  │
│  N/A                                                                                                            │
│                                                                                                                 │
│  **Employees:**                                                                                                 │
│  The employee who has been employed the shortest is Marcus Webb, with a salary of $61,000. He has been          │
│  employed for 1.5 years.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────────────────────────────────────────────────────────────── Tracing Preference Saved ───────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                                                                                                       │
│  Info: Tracing has been disabled.                                                                                                                                                                     │
│                                                                                                                                                                                                       │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                                                                                                       

## 8. Exercises

1. Ask a follow-up question in one `Task` that only needs the employee tool, e.g. *"Who has been
   employed the longest, and what's their salary?"* — notice the agent skips the weather and
   currency tools entirely when they're not relevant.
2. Add a `max_age` filter to `EmployeeLookupInput` and wire it into the SQL query.
3. Swap Open-Meteo for a different weather provider (e.g. OpenWeatherMap, which needs a free API
   key) — only the request/response handling inside `get_weather` changes.
4. Add a fourth tool — e.g. a simple calculator, or a tool that computes average salary per
   department directly with SQL (`GROUP BY department`) — and ask a question that needs it.
5. Try `verbose=False` on the agent and compare how much harder it is to see *why* the agent
   picked the tools it did — verbose logging is your main debugging tool during development.